### Xgboost from PCA v3 dataset 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import random as rd

In [ ]:
df_pca = pd.read_csv('../dataset/full_v3_pca_2026-09-24_22:55:59.csv')
df_pca.head()

In [ ]:
X = df_pca.drop(["Activity", "subject"], axis=1) # Predictors (69 components)
y = df_pca["Activity"]                        # Target variable
le = LabelEncoder()
y = le.fit_transform(y)
y = pd.DataFrame(y)
groups = df_pca["subject"]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=69)

# Get indices for train and test sets based on Subject groups
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

train_subjects = sorted(groups_train.unique())
test_subjects = sorted(groups_test.unique())

print("Subjects in training data:")
print(train_subjects)

print("\nSubjects in test data:")
print(test_subjects)

print("\nNumber of training subjects:", len(train_subjects))
print("Number of test subjects:", len(test_subjects))

overlap = set(train_subjects) & set(test_subjects)
print("\nOverlapping subjects:", overlap)

In [ ]:
param = {
    'colsample_bylevel': np.float64(0.9886757824886667), 
    'colsample_bytree': np.float64(0.5811648359098158), 
    'gamma': np.float64(0.04985829508174178), 
    'learning_rate': np.float64(0.1379485336968442), 
    'max_depth': 3, 
    'min_child_weight': 7, 
    'n_estimators': 182, 
    'reg_alpha': np.float64(0.12695909015950135), 
    'reg_lambda': np.float64(1.7372434109047188), 
    'subsample': np.float64(0.9475977639083295), 
    'random_state': 69}
    
model = xgb.XGBClassifier(**param)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

y_decoded = le.inverse_transform(y_test)
y_pred_decoded = le.inverse_transform(y_pred)
y_decoded = pd.DataFrame(y_decoded)
y_pred_decoded = pd.DataFrame(y_pred_decoded)

print("--- Classification Report (Unseen Subjects) ---")
print(classification_report(y_decoded, y_pred_decoded))

In [ ]:
# Compute normalized matrix (percentages along rows)
cm_normalized = confusion_matrix(y_test, y_pred, normalize='true')

# Get activity labels in the same order used by LabelEncoder
labels = le.classes_

plt.figure(figsize=(10, 7))
sns.heatmap(
    cm_normalized, 
    annot=True, 
    fmt='.1%',              # Format as percentage (e.g., 70.2%)
    cmap='Blues', 
    xticklabels=labels, 
    yticklabels=labels
)

plt.title('Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label (Ground Truth)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()